In [1]:
from pyspark.sql import SparkSession
import os
import sys

In [2]:
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (
    SparkSession.builder
    .master("local[*]")    
    .getOrCreate()
)

In [3]:
df = spark.read.csv("netflix_titles.csv", header=True, inferSchema=True)

df.show(5, truncate=True)

+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami Bouajila, Tr...|         NULL|Septem

In [4]:
shape = df.count(), len(df.columns)

shape

(8809, 12)

In [5]:
df.columns

['show_id',
 'type',
 'title',
 'director',
 'cast',
 'country',
 'date_added',
 'release_year',
 'rating',
 'duration',
 'listed_in',
 'description']

In [6]:
df.summary().show()

+-------+--------------------+-------------+---------------------------------+--------------------+--------------------+----------------+---------------+-----------------+-----------------+-------------+--------------------+--------------------+
|summary|             show_id|         type|                            title|            director|                cast|         country|     date_added|     release_year|           rating|     duration|           listed_in|         description|
+-------+--------------------+-------------+---------------------------------+--------------------+--------------------+----------------+---------------+-----------------+-----------------+-------------+--------------------+--------------------+
|  count|                8809|         8808|                             8807|                6173|                7983|            7977|           8796|             8807|             8803|         8804|                8806|                8806|
|   mean|       

In [7]:
df.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [ ]:
df = df.dropDuplicates()

In [8]:
from pyspark.sql.functions import count

df.count(), df.distinct().count()


(8809, 8809)

In [9]:
df.count(), df.select("show_id").distinct().count()

(8809, 8809)

In [10]:
df.select("type").distinct().show()


+-------------+
|         type|
+-------------+
|      TV Show|
|        Movie|
|William Wyler|
|         NULL|
+-------------+



In [11]:
df.groupBy("country").count().orderBy("count", ascending=False).show()


+--------------------+-----+
|             country|count|
+--------------------+-----+
|       United States| 2805|
|               India|  972|
|                NULL|  832|
|      United Kingdom|  419|
|               Japan|  245|
|         South Korea|  199|
|              Canada|  181|
|               Spain|  145|
|              France|  123|
|              Mexico|  110|
|               Egypt|  106|
|              Turkey|  105|
|             Nigeria|   93|
|           Australia|   87|
|              Taiwan|   81|
|           Indonesia|   79|
|              Brazil|   77|
|United Kingdom, U...|   75|
|         Philippines|   75|
|United States, Ca...|   73|
+--------------------+-----+
only showing top 20 rows


In [12]:
df.select("release_year").distinct().show()


+-----------------+
|     release_year|
+-----------------+
|     Ted Ferguson|
|             1987|
|             1956|
|             2016|
|             2020|
|             2012|
|             1958|
|           40 min|
|             1943|
|             1972|
| Marquell Manning|
|             1988|
|             2019|
|             2017|
|             1977|
|             2014|
|             1971|
|             1984|
|             2013|
|             1982|
+-----------------+
only showing top 20 rows


In [13]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df.columns
])
null_counts.show()


+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|      0|   1|    2|    2636| 826|    832|        13|           2|     6|       5|        3|          3|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+



In [14]:
df.filter((col("title").isNull()) | (col("title") == "")).show()

+--------------------+-------------+-----+-------------+--------------+-------+----------+------------+--------------------+--------------------+---------+-----------+
|             show_id|         type|title|     director|          cast|country|date_added|release_year|              rating|            duration|listed_in|description|
+--------------------+-------------+-----+-------------+--------------+-------+----------+------------+--------------------+--------------------+---------+-----------+
| and probably will."|         NULL| NULL|         NULL|          NULL|   NULL|      NULL|        NULL|                NULL|                NULL|     NULL|       NULL|
|    Flying Fortress"|William Wyler| NULL|United States|March 31, 2017|   1944|     TV-PG|      40 min|Classic Movies, D...|This documentary ...|     NULL|       NULL|
+--------------------+-------------+-----+-------------+--------------+-------+----------+------------+--------------------+--------------------+---------+-----

In [15]:
df = df.fillna({"country": "Unknown"})

In [16]:
df = df.dropna(subset=["date_added", "title"])

In [17]:
df.select("type").distinct().show()

+-------+
|   type|
+-------+
|TV Show|
|  Movie|
+-------+



In [18]:
df.select("duration").distinct().show()


+-----------------+
|         duration|
+-----------------+
|          100 min|
|          153 min|
|           71 min|
|           56 min|
| Donnell Rawlings|
|           13 min|
|          119 min|
|           33 min|
|          165 min|
|       10 Seasons|
|           12 min|
|          204 min|
|          142 min|
|          173 min|
|           27 min|
|          157 min|
|           30 min|
|           39 min|
|        8 Seasons|
|           82 min|
+-----------------+
only showing top 20 rows


In [19]:
from pyspark.sql.functions import col

df = df.filter(
    col("duration").rlike("^[0-9]+ min$") |        
    col("duration").rlike("^[0-9]+ Seasons?$")    
)


In [20]:
df.count()

8774

In [21]:
df.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|            director|                cast|             country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+--------------------+--------------------+--------------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|     Kirsten Johnson|                NULL|       United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|                NULL|Ama Qamata, Khosi...|        South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglan

In [22]:
from pyspark.sql.functions import regexp_replace, trim, col

df = df.withColumn(
    "duration_clean",
    trim(regexp_replace(col("duration"), "[^0-9]", "")) 
)


In [23]:
df.show(5)

+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+--------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|duration_clean|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+--------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|            90|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|             2|
|     s3|TV Show|   

In [24]:
df.groupBy("country").count().orderBy("count", ascending=False).show(truncate=False)

+-----------------------------+-----+
|country                      |count|
+-----------------------------+-----+
|United States                |2796 |
|India                        |972  |
|Unknown                      |829  |
|United Kingdom               |418  |
|Japan                        |244  |
|South Korea                  |199  |
|Canada                       |181  |
|Spain                        |145  |
|France                       |123  |
|Mexico                       |110  |
|Egypt                        |106  |
|Turkey                       |105  |
|Nigeria                      |93   |
|Australia                    |86   |
|Taiwan                       |81   |
|Indonesia                    |79   |
|Brazil                       |77   |
|Philippines                  |75   |
|United Kingdom, United States|75   |
|United States, Canada        |73   |
+-----------------------------+-----+
only showing top 20 rows


In [26]:
df.select("release_year").distinct().show()

+------------+
|release_year|
+------------+
|        1987|
|        1956|
|        2016|
|        2020|
|        2012|
|        1958|
|        1943|
|        1972|
|        1988|
|        2019|
|        2017|
|        1977|
|        2014|
|        1971|
|        1984|
|        2013|
|        1982|
|        2005|
|        2000|
|        1965|
+------------+
only showing top 20 rows


In [27]:
df = df.withColumn('release_year', col('release_year').cast('int'))

In [30]:
df.schema["release_year"].dataType

IntegerType()

In [36]:
df.select("duration_clean").distinct().show()

+--------------+
|duration_clean|
+--------------+
|           148|
|            31|
|            85|
|           137|
|            65|
|            53|
|           133|
|            78|
|           108|
|           155|
|            34|
|           193|
|           115|
|           101|
|           126|
|            81|
|            28|
|            76|
|            26|
|            27|
+--------------+
only showing top 20 rows


In [33]:
df = df.withColumn('duration_clean', col('duration_clean').cast('int'))

In [34]:
dict(df.dtypes)["duration_clean"]


'int'

In [39]:
from pyspark.sql.functions import rank
from pyspark.sql.window import Window

w = Window.partitionBy("release_year").orderBy(col("duration").desc())

df = df.withColumn("rank", rank().over(w)).filter(col("rank") == 1)

df.show()

+-------+-------+--------------------+--------------------+--------------------+--------------------+-----------------+------------+------+---------+--------------------+--------------------+--------------+----+
|show_id|   type|               title|            director|                cast|             country|       date_added|release_year|rating| duration|           listed_in|         description|duration_clean|rank|
+-------+-------+--------------------+--------------------+--------------------+--------------------+-----------------+------------+------+---------+--------------------+--------------------+--------------+----+
|  s4251|TV Show|Pioneers: First W...|                NULL|                NULL|             Unknown|December 30, 2018|        1925| TV-14| 1 Season|            TV Shows|This collection r...|             1|   1|
|  s7791|  Movie|      Prelude to War|         Frank Capra|                NULL|       United States|   March 31, 2017|        1942| TV-14|   52 min|Cla

In [40]:
df.count()

177

In [50]:
from pyspark.sql.functions import col

df.filter(col("release_year") == 2018).show()


+-------+-----+--------------------+--------------------+--------------------+--------------------+-----------------+------------+------+--------+--------------------+--------------------+--------------+----+
|show_id| type|               title|            director|                cast|             country|       date_added|release_year|rating|duration|           listed_in|         description|duration_clean|rank|
+-------+-----+--------------------+--------------------+--------------------+--------------------+-----------------+------------+------+--------+--------------------+--------------------+--------------+----+
|  s1429|Movie| Is Love Enough? Sir|         Rohena Gera|Tillotama Shome, ...|       India, France|  January 8, 2021|        2018| TV-MA|  99 min|Dramas, Independe...|A young widow is ...|            99|   1|
|  s2184|Movie|What Keeps You Alive|       Colin Minihan|Hannah Emily Ande...|              Canada|   August 1, 2020|        2018|     R|  99 min|LGBTQ Movies, Thr.